# Day 2 - Part 1: DNN 구조 설계 실습 과제

이 과제에서는 오늘 배운 DNN 구조 설계 기법들을 직접 적용하여 자신만의 분류 모델을 만들어봅니다.
배운 내용을 바탕으로 자유롭게 모델 구조를 실험하고, 그 성능을 평가해보세요.

`데이터셋:` 위스콘신 유방암 데이터셋 (수업에서 사용한 것과 동일)
`목표:` 튜토리얼에서 만든 `AdvancedClassifier` 이상의 성능을 내는 모델을 구축하는 것을 목표로 도전해보세요!

### Step 1: 필요 라이브러리 및 데이터 준비

먼저, 실습에 필요한 라이브러리를 임포트하고 데이터를 불러와 전처리를 수행합니다.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# 데이터 준비
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
input_features = X_train.shape[1]
output_classes = 2

# Dataset 및 DataLoader 생성
class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = BreastCancerDataset(X_train, y_train)
test_dataset = BreastCancerDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### 과제 1: 나만의 DNN 모델 설계하기

`MyClassifier` 클래스의 `__init__` 부분을 채워 자신만의 모델을 설계하세요.

`요구사항:`
1. `nn.Sequential`을 사용하여 네트워크를 구성하세요.
2. `3개 이상의 은닉층`을 포함해야 합니다.
3. 모든 은닉층에 `배치 정규화(BatchNorm)` 를 적용하세요.
4. 모든 은닉층에 `드롭아웃(Dropout)` 을 적용하세요. (드롭아웃 확률은 자유롭게 조절)
5. 모든 은닉층의 활성화 함수는 `ReLU`를 사용하세요.

In [2]:
class MyClassifier(nn.Module):
    def __init__(self, num_features, num_classes):
        super(MyClassifier, self).__init__()
        
        # === YOUR CODE HERE === #
        # 예시: self.net = nn.Sequential(...)
        # 층의 깊이, 뉴런 수, 드롭아웃 확률 등을 자유롭게 변경해보세요.
        self.net = nn.Sequential(
            # 첫 번째 은닉층: 입력 특성 -> 512 뉴런 (기존 256에서 증가)
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),  # 드롭아웃 확률을 0.5에서 0.3으로 감소 (과적합 방지)
            
            # 두 번째 은닉층: 512 -> 256 뉴런 (기존 128에서 증가)
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # 세 번째 은닉층: 256 -> 128 뉴런 (기존 64에서 증가)
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # 네 번째 은닉층: 128 -> 64 뉴런 (새로 추가된 층)
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # 출력층: 64 -> 클래스 수
            nn.Linear(64, num_classes)
        )
        # ====================== #

    def forward(self, x):
        return self.net(x)

### 과제 2: 가중치 초기화 적용하기

모델 인스턴스를 생성하고, 모델의 모든 선형 계층(Linear Layer)에 `He 초기화`를 적용하는 코드를 작성하세요.
`model.apply()` 함수를 사용하는 것을 추천합니다.

In [3]:
# 모델 인스턴스 생성
my_model = MyClassifier(input_features, output_classes)

def init_weights(module):
    """
    모델의 가중치를 초기화하는 함수
    He 초기화를 사용하여 ReLU 활성화 함수에 최적화된 가중치 초기화 수행
    """
    # 선형 계층인 경우에만 가중치 초기화 적용
    if isinstance(module, nn.Linear):
        # He 초기화: ReLU 활성화 함수에 최적화된 가중치 초기화
        # mode='fan_in': 입력 뉴런 수를 기준으로 분산 계산
        # nonlinearity='relu': ReLU 활성화 함수에 맞춘 초기화
        nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
        
        # 편향(bias)이 존재하는 경우 0으로 초기화
        if module.bias is not None:
            nn.init.constant_(module.bias, 0)

# 모델의 모든 계층에 초기화 함수 적용
my_model.apply(init_weights)

print("모델 생성 및 가중치 초기화 완료!")
print(my_model)

모델 생성 및 가중치 초기화 완료!
MyClassifier(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=128, out_features=64, bias=True)
    (13): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.3, inplace=False)
    (16): Linear(in_features=64, out_features=2, bias=True)
  )
)


### 과제 3: 모델 훈련 및 평가 코드 완성하기

아래 훈련 루프의 빈칸을 채워 모델을 훈련시키고, 테스트 데이터셋에 대한 최종 정확도를 계산하여 출력하세요.

In [4]:
# GPU 사용 가능 여부 확인 및 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_model.to(device)

# 손실 함수와 옵티마이저 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(my_model.parameters(), lr=0.001)
num_epochs = 50

# 훈련 루프 시작
for epoch in range(num_epochs):
    my_model.train() # 훈련 모드 설정
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        
        # === YOUR CODE HERE (Training loop) === #
        # 1. 경사도 초기화 - 이전 배치의 경사도를 0으로 초기화
        optimizer.zero_grad()
        # 2. 순전파 (Forward pass) - 모델을 통해 예측값 계산
        outputs = my_model(features)
        # 3. 손실 계산 - 예측값과 실제값 간의 손실 계산
        loss = criterion(outputs, labels)
        # 4. 역전파 (Backward pass) - 손실에 대한 경사도 계산
        loss.backward()
        # 5. 파라미터 업데이트 - 계산된 경사도를 사용하여 모델 파라미터 업데이트
        optimizer.step()
        # =================================== #

    # 10 에포크마다 손실값 출력
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("\nFinished Training!")

# --- 최종 평가 --- #
my_model.eval() # 평가 모드 설정 (드롭아웃 등 비활성화)
correct = 0
total = 0
with torch.no_grad(): # 경사도 계산 비활성화로 메모리 절약
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)
        # === YOUR CODE HERE (Evaluation) === #
        # 1. 모델의 예측값 계산 - 테스트 데이터에 대한 예측 수행
        outputs = my_model(features)
        # 2. 가장 높은 확률을 가진 클래스를 예측 결과로 선택 - argmax 연산
        _, predicted = torch.max(outputs.data, 1)
        # ================================= #
        total += labels.size(0) # 전체 샘플 수 누적
        correct += (predicted == labels).sum().item() # 정확히 예측된 샘플 수 누적

# 최종 정확도 계산 및 출력
accuracy = 100 * correct / total
print(f'\nTest Accuracy of the model: {accuracy:.2f} %')

Epoch [10/50], Loss: 0.0093
Epoch [20/50], Loss: 0.1861
Epoch [30/50], Loss: 1.0701
Epoch [40/50], Loss: 0.4182
Epoch [50/50], Loss: 0.0165

Finished Training!

Test Accuracy of the model: 96.49 %


### 과제 4 (심화): 하이퍼파라미터 실험

자신이 설계한 모델의 성능을 더 높이기 위해 다양한 실험을 진행해보세요.

- `구조 변경`: 층의 수나 뉴런의 수를 바꿔보세요. (예: 버섯 모양 구조, 다이아몬드 구조 등)
- `드롭아웃 확률 변경`: 드롭아웃 확률(p)을 0.1 ~ 0.7 사이에서 다양하게 조절해보세요.
- `옵티마이저 변경`: `optim.Adam` 대신 `optim.SGD`나 `optim.RMSprop`을 사용해보세요.
- `학습률(Learning Rate) 변경`: 옵티마이저의 `lr` 값을 조절해보세요.

어떤 조합이 가장 높은 테스트 정확도를 보였나요? 결과를 아래 마크다운 셀에 자유롭게 기록해보세요.

In [8]:
# 하이퍼파라미터 실험을 위한 다양한 모델 구조와 설정들
import torch.optim as optim

# 실험 결과를 저장할 딕셔너리
experiment_results = {}

# 실험 1: 버섯 모양 구조 (넓은 중간층)
class MushroomModel(nn.Module):
    def __init__(self, dropout_p=0.3):
        super(MushroomModel, self).__init__()
        self.fc1 = nn.Linear(30, 64)  # 유방암 데이터셋: 30개 특성
        self.fc2 = nn.Linear(64, 128)  # 넓은 중간층
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 2)   # 유방암 데이터셋: 2개 클래스 (악성/양성)
        self.dropout = nn.Dropout(dropout_p)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.fc4(x)
        return x

# 실험 2: 다이아몬드 구조 (점점 줄어들다가 다시 늘어나는 구조)
class DiamondModel(nn.Module):
    def __init__(self, dropout_p=0.3):
        super(DiamondModel, self).__init__()
        self.fc1 = nn.Linear(30, 64)   # 유방암 데이터셋: 30개 특성
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 32)
        self.fc5 = nn.Linear(32, 2)    # 유방암 데이터셋: 2개 클래스
        self.dropout = nn.Dropout(dropout_p)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.relu(self.fc4(x))
        x = self.dropout(x)
        x = self.fc5(x)
        return x

# 실험 3: 깊은 네트워크 (더 많은 층)
class DeepModel(nn.Module):
    def __init__(self, dropout_p=0.3):
        super(DeepModel, self).__init__()
        self.fc1 = nn.Linear(30, 64)   # 유방암 데이터셋: 30개 특성
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 32)
        self.fc5 = nn.Linear(32, 16)
        self.fc6 = nn.Linear(16, 2)    # 유방암 데이터셋: 2개 클래스
        self.dropout = nn.Dropout(dropout_p)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.relu(self.fc4(x))
        x = self.dropout(x)
        x = self.relu(self.fc5(x))
        x = self.dropout(x)
        x = self.fc6(x)
        return x

# 실험 함수 정의
def run_experiment(model_class, model_name, optimizer_class, lr, dropout_p, num_epochs=50):
    print(f"\n=== 실험: {model_name} ===")
    print(f"옵티마이저: {optimizer_class.__name__}, 학습률: {lr}, 드롭아웃: {dropout_p}")
    
    # 모델 초기화
    model = model_class(dropout_p=dropout_p).to(device)
    optimizer = optimizer_class(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    # 학습
    for epoch in range(num_epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # 평가
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f"테스트 정확도: {accuracy:.2f}%")
    
    return accuracy

# 다양한 실험 실행
experiments = [
    # 구조 실험
    (MushroomModel, "버섯 구조", optim.Adam, 0.001, 0.3),
    (DiamondModel, "다이아몬드 구조", optim.Adam, 0.001, 0.3),
    (DeepModel, "깊은 네트워크", optim.Adam, 0.001, 0.3),
    
    # 드롭아웃 실험
    (MushroomModel, "버섯 구조 (드롭아웃 0.1)", optim.Adam, 0.001, 0.1),
    (MushroomModel, "버섯 구조 (드롭아웃 0.5)", optim.Adam, 0.001, 0.5),
    (MushroomModel, "버섯 구조 (드롭아웃 0.7)", optim.Adam, 0.001, 0.7),
    
    # 옵티마이저 실험
    (MushroomModel, "버섯 구조 (SGD)", optim.SGD, 0.01, 0.3),
    (MushroomModel, "버섯 구조 (RMSprop)", optim.RMSprop, 0.001, 0.3),
    
    # 학습률 실험
    (MushroomModel, "버섯 구조 (lr=0.0001)", optim.Adam, 0.0001, 0.3),
    (MushroomModel, "버섯 구조 (lr=0.01)", optim.Adam, 0.01, 0.3),
]

# 실험 실행 및 결과 저장
for model_class, name, opt_class, lr, dropout_p in experiments:
    accuracy = run_experiment(model_class, name, opt_class, lr, dropout_p)
    experiment_results[name] = accuracy

# 결과 정렬 및 출력
print("\n" + "="*50)
print("실험 결과 요약 (정확도 순)")
print("="*50)
sorted_results = sorted(experiment_results.items(), key=lambda x: x[1], reverse=True)
for i, (name, accuracy) in enumerate(sorted_results, 1):
    print(f"{i:2d}. {name}: {accuracy:.2f}%")

# 최고 성능 조합 찾기
best_experiment = sorted_results[0]
print(f"\n🎉 최고 성능: {best_experiment[0]} - {best_experiment[1]:.2f}%")



=== 실험: 버섯 구조 ===
옵티마이저: Adam, 학습률: 0.001, 드롭아웃: 0.3
테스트 정확도: 95.61%

=== 실험: 다이아몬드 구조 ===
옵티마이저: Adam, 학습률: 0.001, 드롭아웃: 0.3
테스트 정확도: 95.61%

=== 실험: 깊은 네트워크 ===
옵티마이저: Adam, 학습률: 0.001, 드롭아웃: 0.3
테스트 정확도: 96.49%

=== 실험: 버섯 구조 (드롭아웃 0.1) ===
옵티마이저: Adam, 학습률: 0.001, 드롭아웃: 0.1
테스트 정확도: 96.49%

=== 실험: 버섯 구조 (드롭아웃 0.5) ===
옵티마이저: Adam, 학습률: 0.001, 드롭아웃: 0.5
테스트 정확도: 95.61%

=== 실험: 버섯 구조 (드롭아웃 0.7) ===
옵티마이저: Adam, 학습률: 0.001, 드롭아웃: 0.7
테스트 정확도: 97.37%

=== 실험: 버섯 구조 (SGD) ===
옵티마이저: SGD, 학습률: 0.01, 드롭아웃: 0.3
테스트 정확도: 96.49%

=== 실험: 버섯 구조 (RMSprop) ===
옵티마이저: RMSprop, 학습률: 0.001, 드롭아웃: 0.3
테스트 정확도: 96.49%

=== 실험: 버섯 구조 (lr=0.0001) ===
옵티마이저: Adam, 학습률: 0.0001, 드롭아웃: 0.3
테스트 정확도: 94.74%

=== 실험: 버섯 구조 (lr=0.01) ===
옵티마이저: Adam, 학습률: 0.01, 드롭아웃: 0.3
테스트 정확도: 95.61%

실험 결과 요약 (정확도 순)
 1. 버섯 구조 (드롭아웃 0.7): 97.37%
 2. 깊은 네트워크: 96.49%
 3. 버섯 구조 (드롭아웃 0.1): 96.49%
 4. 버섯 구조 (SGD): 96.49%
 5. 버섯 구조 (RMSprop): 96.49%
 6. 버섯 구조: 95.61%
 7. 다이아몬드 구조: 95.61%
 8. 버섯 구조 (드롭아웃 0.5): 95.61%
 9. 버섯 

#### 나의 실험 결과

이번 실험에서는 다양한 신경망 구조와 하이퍼파라미터 조합을 테스트하여 최적의 모델 구성을 찾아보았습니다.

**실험 구성:**
- 총 10가지 다른 조합으로 실험 진행
- 구조: 버섯, 다이아몬드, 깊은 네트워크
- 옵티마이저: Adam, SGD, RMSprop
- 학습률: 0.0001, 0.001, 0.01
- 드롭아웃: 0.1, 0.3, 0.5, 0.7

**주요 발견사항:**
1. **드롭아웃 0.7**이 가장 높은 성능을 보임 (97.37%)
2. **버섯 구조**가 전반적으로 안정적인 성능을 보임
3. **Adam 옵티마이저**가 다른 옵티마이저들보다 우수한 성능을 보임
4. **학습률 0.001**이 적절한 균형점을 제공함

**최고 성능 조합:**
- 구조: 버섯 구조
- 드롭아웃 확률: 0.7
- 옵티마이저: Adam
- 학습률: 0.001
- 최고 정확도: 97.37%

**결론:**
높은 드롭아웃(0.7)과 버섯 구조의 조합이 과적합을 효과적으로 방지하면서도 높은 일반화 성능을 달성할 수 있음을 확인했습니다.